In [2]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import os
from concurrent.futures import ThreadPoolExecutor

# إنشاء مجلد لحفظ الصور
os.makedirs("images", exist_ok=True)

books = []

def parse_page(page_num):
    url = f"https://books.toscrape.com/catalogue/page-{page_num}.html"
    response = requests.get(url, timeout=10)
    soup = BeautifulSoup(response.text, "html.parser")
    items = soup.find_all("article", class_="product_pod")

    local_books = []
    for item in items:
        name = item.h3.a["title"]
        price = item.find("p", class_="price_color").text
        rating = item.p["class"][1]
        stars = {"One":"⭐","Two":"⭐⭐","Three":"⭐⭐⭐","Four":"⭐⭐⭐⭐","Five":"⭐⭐⭐⭐⭐"}
        rating = stars.get(rating, rating)
        link = "https://books.toscrape.com/catalogue/" + item.h3.a["href"]
        image_url = "https://books.toscrape.com/" + item.find("img")["src"].replace("../","")

        # تحميل الصورة
        image_name = os.path.join("images", name.replace(" ","_") + ".jpg")
        try:
            img_data = requests.get(image_url, timeout=10).content
            with open(image_name, "wb") as f:
                f.write(img_data)
        except:
            image_name = "Failed to download"

        local_books.append({
            "Name": name,
            "Price": price,
            "Rating": rating,
            "Link": link,
            "Image Path": image_name
        })

    return local_books

# تشغيل Scraper بخيوط Threads لتسريع التحميل
with ThreadPoolExecutor(max_workers=10) as executor:
    results = executor.map(parse_page, range(1, 51))

# دمج النتائج
for r in results:
    books.extend(r)

# حفظ البيانات في Excel
df = pd.DataFrame(books)
df.to_excel("books_portfolio.xlsx", index=False)

print("✅ Scraper Ultimate انتهى! الملف والصور جاهزين في المجلد الحالي.")

✅ Scraper Ultimate انتهى! الملف والصور جاهزين في المجلد الحالي.


In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import os
from concurrent.futures import ThreadPoolExecutor

# إنشاء مجلد لحفظ الصور (اختياري: يمكنك حذفه إذا لم تعد تريد حفظ الصور على جهازك)
os.makedirs("images", exist_ok=True)

def parse_page(page_num):
    url = f"https://books.toscrape.com/catalogue/page-{page_num}.html"
    try:
        response = requests.get(url, timeout=10)
    except:
        return []
        
    soup = BeautifulSoup(response.text, "html.parser")
    items = soup.find_all("article", class_="product_pod")

    local_books = []
    for item in items:
        name = item.h3.a["title"]
        price = item.find("p", class_="price_color").text
        rating_class = item.p["class"][1]
        stars = {"One":"⭐","Two":"⭐⭐","Three":"⭐⭐⭐","Four":"⭐⭐⭐⭐","Five":"⭐⭐⭐⭐⭐"}
        rating = stars.get(rating_class, rating_class)
        link = "https://books.toscrape.com/catalogue/" + item.h3.a["href"]
        
        # 🌟 استخراج رابط الصورة المباشر من الإنترنت
        image_url = "https://books.toscrape.com/" + item.find("img")["src"].replace("../", "")

        # تنظيف اسم الصورة من الرموز التي قد تسبب مشاكل في الويندوز
        safe_name = name.replace(" ", "_").replace("/", "_").replace("\\", "_").replace(":", "")
        image_name = os.path.join("images", safe_name + ".jpg")
        
        # تحميل الصورة محلياً
        try:
            img_data = requests.get(image_url, timeout=10).content
            with open(image_name, "wb") as f:
                f.write(img_data)
        except:
            image_name = "Failed to download"

        local_books.append({
            "Name": name,
            "Price": price,
            "Rating": rating,
            "Link": link,
            "Image URL": image_url,  # 👈 هذا هو العمود السحري الذي سيستخدمه Streamlit
            "Image Path": image_name # هذا المسار المحلي على جهازك
        })

    return local_books

books = []
print("⏳ جاري سحب البيانات، يرجى الانتظار...")

# تشغيل Scraper بخيوط Threads لتسريع التحميل
with ThreadPoolExecutor(max_workers=10) as executor:
    results = executor.map(parse_page, range(1, 51))

# دمج النتائج
for r in results:
    if r:
        books.extend(r)

# حفظ البيانات في Excel (ملف واحد يكفي)
df = pd.DataFrame(books)
df.to_excel("books_portfolio.xlsx", index=False)

print("✅ السحب انتهى بنجاح! الملف جاهز ويحتوي على روابط الإنترنت المباشرة.")

⏳ جاري سحب البيانات، يرجى الانتظار...
✅ السحب انتهى بنجاح! الملف جاهز ويحتوي على روابط الإنترنت المباشرة.


In [2]:
from openpyxl import load_workbook

wb = load_workbook("books_portfolio.xlsx")
ws = wb.active

# ضبط عرض الأعمدة
ws.column_dimensions["A"].width = 40
ws.column_dimensions["B"].width = 10
ws.column_dimensions["C"].width = 15
ws.column_dimensions["D"].width = 50
ws.column_dimensions["E"].width = 50

wb.save("books_portfolio.xlsx")